# Positional Embeddings in Large Language Models
## A Comprehensive Guide to Encoding Sequential Order in Transformer Architectures

---

### Motivation

The Transformer architecture (Vaswani et al., 2017) revolutionised natural language processing by replacing recurrence with **self-attention**. However, self-attention is fundamentally a **set operation** — it computes pairwise interactions between all tokens without any inherent notion of order. Given a sequence $\{x_1, x_2, \ldots, x_n\}$, a vanilla self-attention layer produces the same output regardless of how tokens are permuted.

This is a critical problem: the sentence *"The cat sat on the mat"* has a fundamentally different meaning from *"The mat sat on the cat"*. Without positional information, a Transformer cannot distinguish between these two sequences.

**Positional embeddings** solve this by injecting information about each token's absolute or relative position into the model, enabling it to reason about sequential structure.

### Taxonomy of Positional Encoding Methods

| Generation | Method | Injection Point | Key Innovation |
| --- | --- | --- | --- |
| 1st | Sinusoidal (Fixed) | Added to input embeddings | Frequency-based, no learnable params |
| 1st | Learned Absolute | Added to input embeddings | Fully learnable position vectors |
| 2nd | Relative (Shaw et al.) | Added to attention logits | Encodes pairwise distances |
| 3rd | RoPE | Rotates Q/K vectors | Relative via absolute rotation |
| 3rd | ALiBi | Biases attention scores | No positional embedding at all |

We will examine each method in depth: the mathematical formulation, PyTorch implementation, and critical analysis of strengths and limitations.

## 1. The Permutation Invariance Problem

### Self-Attention Recap

Given input embeddings $$X \in \mathbb{R}^{n \times d}$$ (where $$n$$ is sequence length and $$d$$ is model dimension), self-attention computes:

$$
Q = XW_Q, \quad K = XW_K, \quad V = XW_V
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

### Why This Is Position-Agnostic

Let $$\Pi$$ be any permutation matrix. If we permute the input $$X' = \Pi X$$, then:

$$
Q' = \Pi X W_Q = \Pi Q
$$

$$
\text{Attention}(Q', K', V') = \Pi \cdot \text{Attention}(Q, K, V)
$$

The output is simply the **same permutation** applied to the original output. The model treats the input as a **bag of tokens** — it cannot tell that token at position 3 came before token at position 7.

### What Positional Embeddings Must Provide

An effective positional encoding scheme should satisfy:

1. **Uniqueness**: Each position gets a distinct encoding
2. **Determinism**: The same position always gets the same encoding
3. **Bounded values**: Encodings should not grow unboundedly with sequence length
4. **Generalisation**: Ideally, the model should handle sequences longer than those seen in training
5. **Relative distance awareness**: The model should be able to infer that positions 3 and 5 are the same distance apart as positions 10 and 12

## 2. Sinusoidal (Fixed) Positional Embeddings
*Introduced in: "Attention Is All You Need" (Vaswani et al., 2017)*

---

### Intuition

The core idea is to represent each position as a unique point in $$d$$-dimensional space using **sinusoidal functions of varying frequencies**. Just as a Fourier series can represent any periodic signal as a sum of sines and cosines, each position is encoded by a unique combination of sinusoidal values.

Think of it like a **binary clock**: the least significant bit oscillates every step, the next bit every 2 steps, the next every 4 steps, etc. Sinusoidal encodings are a continuous, smooth analogue of this idea.

### Mathematical Formulation

For position $$pos$$ and dimension index $$i$$ (where $$0 \le i < d/2$$):

$$
PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d}}\right)
$$

$$
PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d}}\right)
$$

Alternatively, defining the frequency $$\omega_i = \frac{1}{10000^{2i/d}}$$:

$$
PE(pos) = \left[\sin(\omega_0 \cdot pos), \cos(\omega_0 \cdot pos), \sin(\omega_1 \cdot pos), \cos(\omega_1 \cdot pos), \ldots, \sin(\omega_{d/2-1} \cdot pos), \cos(\omega_{d/2-1} \cdot pos)\right]
$$

### Key Property: Relative Position via Linear Transformation

A crucial property is that for any fixed offset $$k$$, there exists a linear transformation $$M_k$$ such that:

$$
PE(pos + k) = M_k \cdot PE(pos)
$$

This is because:

$$
\begin{bmatrix} \sin(\omega(pos+k)) \\ \cos(\omega(pos+k)) \end{bmatrix} = \begin{bmatrix} \cos(\omega k) & \sin(\omega k) \\ -\sin(\omega k) & \cos(\omega k) \end{bmatrix} \begin{bmatrix} \sin(\omega \cdot pos) \\ \cos(\omega \cdot pos) \end{bmatrix}
$$

This rotation matrix $$M_k$$ depends only on the offset $$k$$, not on the absolute position. This means the model can **learn to attend to relative positions** through the dot-product attention mechanism.

### How It's Applied

The positional encoding is **added** to the token embeddings before being fed into the first Transformer layer:

$$
\hat{X} = X + PE
$$

where $$X \in \mathbb{R}^{n \times d}$$ is the token embedding matrix and $$PE \in \mathbb{R}^{n \times d}$$ is the positional encoding matrix.

### Pros

* **No learnable parameters** — reduces model size and avoids overfitting positional patterns
* **Theoretically supports infinite sequence lengths** — can generate encodings for any position
* **Smooth, continuous representation** — nearby positions have similar encodings (graceful degradation)
* **Linear transformation property** enables learning relative positions implicitly

### Cons

* **Empirically weaker** than learned embeddings on many benchmarks (the model cannot adapt positional representations to the task)
* **Information decay across layers** — positional signal added only at the input diminishes as it propagates through deep networks
* **Fixed frequency spectrum** — the geometric progression of wavelengths may not be optimal for all tasks
* **Extrapolation is not free** — despite being defined for all positions, performance still degrades beyond training-length sequences because the model has never learned to use those patterns

In [0]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
import numpy as np

class SinusoidalPositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding as described in 'Attention Is All You Need'.
    
    Generates fixed (non-learnable) positional encodings using sine and cosine
    functions of different frequencies.
    """
    
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create positional encoding matrix [max_len, d_model]
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # [max_len, 1]
        
        # Compute the division term: 10000^(2i/d_model)
        # Using log-space for numerical stability:
        # exp(2i * (-log(10000) / d_model)) = 1 / 10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )  # [d_model/2]
        
        # Apply sin to even indices, cos to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)  # Even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)  # Odd dimensions
        
        # Add batch dimension: [1, max_len, d_model]
        pe = pe.unsqueeze(0)
        
        # Register as buffer (not a parameter, but moves with the model)
        self.register_buffer('pe', pe)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Token embeddings [batch_size, seq_len, d_model]
        Returns:
            Position-encoded embeddings [batch_size, seq_len, d_model]
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# --- Demonstration ---
d_model = 128
max_len = 100

encoder = SinusoidalPositionalEncoding(d_model=d_model, max_len=max_len, dropout=0.0)

# Generate encodings for visualization
positions = torch.zeros(1, max_len, d_model)  # Dummy input
pe_values = encoder.pe[0, :max_len, :].numpy()  # Extract the PE matrix

print(f"Positional Encoding shape: {pe_values.shape}")
print(f"PE[0, :8] = {pe_values[0, :8]}")
print(f"PE[1, :8] = {pe_values[1, :8]}")
print(f"\nNote: Even dims use sin, odd dims use cos")
print(f"Position 0, dim 0 (sin): {pe_values[0, 0]:.4f}")
print(f"Position 0, dim 1 (cos): {pe_values[0, 1]:.4f}")

# Visualize the positional encodings
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Heatmap of full PE matrix
ax = axes[0, 0]
im = ax.imshow(pe_values, cmap='RdBu', aspect='auto', interpolation='nearest')
ax.set_xlabel('Embedding Dimension')
ax.set_ylabel('Position')
ax.set_title('Sinusoidal Positional Encoding Heatmap')
plt.colorbar(im, ax=ax)

# Individual dimension curves
ax = axes[0, 1]
for dim in [0, 1, 4, 5, 20, 21, 60, 61]:
    ax.plot(pe_values[:, dim], label=f'dim {dim}', alpha=0.7)
ax.set_xlabel('Position')
ax.set_ylabel('Encoding Value')
ax.set_title('PE Values Across Positions (Selected Dimensions)')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

# Dot product between position 0 and all others (shows relative distance)
ax = axes[1, 0]
pe_tensor = torch.tensor(pe_values)
dot_products = torch.matmul(pe_tensor, pe_tensor[0])  # Similarity to position 0
ax.plot(dot_products.numpy())
ax.set_xlabel('Position')
ax.set_ylabel('Dot Product with Position 0')
ax.set_title('Positional Similarity (Dot Product with pos=0)')
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

# Distance matrix
ax = axes[1, 1]
similarity_matrix = torch.matmul(pe_tensor[:50], pe_tensor[:50].T)
im = ax.imshow(similarity_matrix.numpy(), cmap='viridis', aspect='auto')
ax.set_xlabel('Position')
ax.set_ylabel('Position')
ax.set_title('Pairwise Dot-Product Similarity (First 50 Positions)')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print("\n✓ Sinusoidal PE successfully implemented and visualized")

## 3. Learned (Absolute) Positional Embeddings
*Used in: BERT (Devlin et al., 2019), GPT-2 (Radford et al., 2019), GPT-3*

---

### Intuition

Instead of using a fixed mathematical formula, we let the model **learn** the optimal positional representation during training. Each position $$pos \in \{0, 1, \ldots, L_{max}-1\}$$ is assigned a learnable embedding vector $$p_{pos} \in \mathbb{R}^d$$, stored in a lookup table (an embedding matrix).

### Mathematical Formulation

Define a learnable embedding matrix:

$$
P \in \mathbb{R}^{L_{max} \times d}
$$

where $$L_{max}$$ is the maximum sequence length supported by the model and $$d$$ is the model dimension.

For a token at position $$pos$$:

$$
\hat{x}_{pos} = x_{pos} + P[pos]
$$

Or in matrix form for the entire sequence:

$$
\hat{X} = X + P[:n, :]
$$

where $$n$$ is the actual sequence length ($$n \le L_{max}$$).

### Training Dynamics

During pre-training, the position embeddings $$P$$ are initialised randomly (typically from $$\mathcal{N}(0, 0.02)$$) and updated via backpropagation just like any other model parameter. The gradient signal from the language modelling objective shapes the positional representations to capture:

* **Local patterns** (adjacent token relationships)
* **Syntactic distances** (subject-verb agreement across clauses)
* **Semantic roles** (beginning vs. end of sequences)

### Pros

* **Empirically strong** — consistently outperforms sinusoidal encodings on downstream tasks (BERT, GPT-2 ablations confirm this)
* **Task-adaptive** — the model learns position representations that are optimal for its training objective
* **Simple implementation** — just an `nn.Embedding` layer
* **Can capture non-trivial positional patterns** — not constrained to any particular functional form

### Cons

* **Hard length limit** — cannot process sequences longer than $$L_{max}$$; any position beyond the table simply has no embedding
* **No extrapolation** — completely fails at positions beyond training length (unlike sinusoidal which at least produces *some* value)
* **Extra parameters** — adds $$L_{max} \times d$$ parameters (e.g., for GPT-2: 1024 × 768 = 786,432 params)
* **Positional signal decay** — like sinusoidal, it's added only at the input and degrades through deep layers
* **No inherent notion of relative distance** — the model must *learn* that position 5 and 7 have the same relationship as position 12 and 14; nothing in the architecture guarantees this

In [0]:
class LearnedPositionalEmbedding(nn.Module):
    """
    Learned Positional Embeddings as used in BERT and GPT-2.
    
    Each position has a unique learnable embedding vector that is optimised
    during training via backpropagation.
    """
    
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Learnable embedding table: each position gets a d_model-dimensional vector
        self.position_embeddings = nn.Embedding(max_len, d_model)
        
        # Register a position index buffer for easy slicing
        self.register_buffer(
            'position_ids',
            torch.arange(max_len).unsqueeze(0)  # [1, max_len]
        )
        
        # Initialise with small random values (following BERT)
        nn.init.normal_(self.position_embeddings.weight, mean=0.0, std=0.02)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Token embeddings [batch_size, seq_len, d_model]
        Returns:
            Position-encoded embeddings [batch_size, seq_len, d_model]
        """
        seq_len = x.size(1)
        position_ids = self.position_ids[:, :seq_len]  # [1, seq_len]
        position_embeddings = self.position_embeddings(position_ids)  # [1, seq_len, d_model]
        x = x + position_embeddings
        return self.dropout(x)


# --- Demonstration ---
d_model = 128
max_len = 512

learned_pe = LearnedPositionalEmbedding(d_model=d_model, max_len=max_len, dropout=0.0)

# Show the initialised embeddings
with torch.no_grad():
    # Dummy token embeddings
    dummy_tokens = torch.randn(2, 50, d_model)  # batch=2, seq_len=50
    output = learned_pe(dummy_tokens)
    
    print(f"Input shape:  {dummy_tokens.shape}")
    print(f"Output shape: {output.shape}")
    print(f"\nParameter count: {sum(p.numel() for p in learned_pe.parameters()):,}")
    print(f"  = max_len ({max_len}) × d_model ({d_model}) = {max_len * d_model:,}")

# Visualise the (randomly initialised) learned embeddings
pe_weights = learned_pe.position_embeddings.weight.detach().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Heatmap
ax = axes[0]
im = ax.imshow(pe_weights[:100, :], cmap='RdBu', aspect='auto')
ax.set_xlabel('Dimension')
ax.set_ylabel('Position')
ax.set_title('Learned PE Weights (Random Init)')
plt.colorbar(im, ax=ax)

# Compare with sinusoidal - similarity structure
ax = axes[1]
learned_tensor = torch.tensor(pe_weights[:50])
sim_learned = torch.matmul(learned_tensor, learned_tensor.T)
im = ax.imshow(sim_learned.numpy(), cmap='viridis', aspect='auto')
ax.set_xlabel('Position')
ax.set_ylabel('Position')
ax.set_title('Learned PE: Pairwise Similarity (Init)')
plt.colorbar(im, ax=ax)

# What learned embeddings look like AFTER training (simulated with known structure)
# We simulate what a trained model's position embeddings might look like
ax = axes[2]
# After training, nearby positions tend to be similar
trained_sim = np.zeros((50, 50))
for i in range(50):
    for j in range(50):
        trained_sim[i, j] = np.exp(-abs(i - j) / 10)  # Exponential decay with distance
im = ax.imshow(trained_sim, cmap='viridis', aspect='auto')
ax.set_xlabel('Position')
ax.set_ylabel('Position')
ax.set_title('Typical Trained PE: Similarity (Illustrative)')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print("\n✓ Learned Positional Embedding implemented")
print("\nKey difference from sinusoidal:")
print("  - At init: random, no structure")
print("  - After training: learns distance-based similarity patterns")
print("  - Trade-off: better performance but fixed max length")

## 4. Relative Positional Embeddings
*Introduced in: "Self-Attention with Relative Position Representations" (Shaw et al., 2018)*
*Extended in: Transformer-XL (Dai et al., 2019), T5 (Raffel et al., 2020)*

---

### Motivation

Absolute position encodings have a fundamental limitation: they encode **where** a token is, but linguistic relationships depend on **how far apart** tokens are. The word "not" modifies meaning based on its distance to the verb, not its absolute position in the sentence.

Relative positional embeddings inject positional information directly into the **attention mechanism** rather than the input embeddings, encoding the pairwise distance between every query and key.

### Mathematical Formulation (Shaw et al., 2018)

In standard self-attention, the attention logit between positions $$i$$ and $$j$$ is:

$$
e_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}}
$$

Shaw et al. augment this with learnable relative position embeddings:

$$
e_{ij} = \frac{q_i \cdot (k_j + a_{ij}^K)}{\sqrt{d_k}}
$$

where $$a_{ij}^K \in \mathbb{R}^{d_k}$$ is a learnable embedding that depends only on the **relative position** $$(j - i)$$.

Similarly, the value computation is augmented:

$$
z_i = \sum_j \alpha_{ij} (v_j + a_{ij}^V)
$$

where $$a_{ij}^V \in \mathbb{R}^{d_v}$$ is another relative position embedding for values.

### Clipping

To keep the embedding table finite, relative distances are clipped to a maximum range:

$$
a_{ij}^K = w_{\text{clip}(j-i, -k, k)}^K
$$

where $$\text{clip}(x, a, b) = \max(a, \min(b, x))$$. This means positions farther than $$k$$ apart share the same embedding, encoding "far away" as a single concept.

### T5's Relative Position Bias (Simplified)

T5 (Raffel et al., 2020) uses an even simpler version: a **scalar bias** per relative position per head:

$$
e_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}} + b(i - j)
$$

where $$b: \mathbb{Z} \rightarrow \mathbb{R}$$ is a learnable scalar function of relative distance, separate per attention head. T5 uses logarithmic bucketing to map large distances into a fixed number of bins.

### Pros

* **Explicitly encodes relative distance** — the model directly sees how far apart tokens are, rather than having to infer it
* **Translation invariant** — the relationship between positions 3,7 is identical to 10,14
* **Better generalisation to longer sequences** than absolute methods (especially T5's version)
* **Per-layer injection** — positional information is fresh at every layer, avoiding the signal decay problem

### Cons

* **Computational overhead** — adds extra operations to every attention computation (not just at the input)
* **Memory cost** — must store or compute the relative position bias matrix of size $$O(n^2)$$ per layer per head
* **Clipping loses fine-grained long-range information** — all positions beyond the clip distance look the same
* **Increased complexity** — significantly more complex implementation than absolute methods
* **Still fundamentally bounded** — while generalisation is better, it's not unlimited

In [0]:
class RelativePositionEmbedding(nn.Module):
    """
    Relative Position Embeddings following Shaw et al. (2018).
    
    Adds learnable relative position embeddings to attention logits.
    Distances are clipped to [-max_relative_position, max_relative_position].
    """
    
    def __init__(self, d_model: int, num_heads: int, max_relative_position: int = 32):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.max_relative_position = max_relative_position
        
        # Total number of relative position buckets: 2*k + 1 (negative, zero, positive)
        num_embeddings = 2 * max_relative_position + 1
        
        # Learnable relative position embeddings for keys
        self.relative_position_embeddings_k = nn.Embedding(num_embeddings, self.d_head)
        
        # Learnable relative position embeddings for values
        self.relative_position_embeddings_v = nn.Embedding(num_embeddings, self.d_head)
        
        nn.init.xavier_uniform_(self.relative_position_embeddings_k.weight)
        nn.init.xavier_uniform_(self.relative_position_embeddings_v.weight)
    
    def _get_relative_positions(self, seq_len: int) -> torch.Tensor:
        """
        Compute the clipped relative position matrix.
        
        Returns:
            Tensor of shape [seq_len, seq_len] with clipped relative positions
            shifted to be non-negative indices into the embedding table.
        """
        # Create position indices
        positions = torch.arange(seq_len)  # [seq_len]
        
        # Compute pairwise relative positions: relative_pos[i, j] = j - i
        relative_positions = positions.unsqueeze(0) - positions.unsqueeze(1)  # [seq_len, seq_len]
        
        # Clip to [-max_relative_position, max_relative_position]
        relative_positions = torch.clamp(
            relative_positions, 
            -self.max_relative_position, 
            self.max_relative_position
        )
        
        # Shift to non-negative indices: [0, 2*max_relative_position]
        relative_positions = relative_positions + self.max_relative_position
        
        return relative_positions
    
    def forward(
        self, 
        queries: torch.Tensor, 
        keys: torch.Tensor, 
        values: torch.Tensor
    ) -> tuple:
        """
        Compute relative position-augmented attention.
        
        Args:
            queries: [batch, num_heads, seq_len, d_head]
            keys:    [batch, num_heads, seq_len, d_head]
            values:  [batch, num_heads, seq_len, d_head]
        Returns:
            (attention_output, attention_weights)
        """
        batch_size, num_heads, seq_len, d_head = queries.shape
        
        # Standard attention logits: Q @ K^T
        attn_logits = torch.matmul(queries, keys.transpose(-2, -1))  # [B, H, N, N]
        
        # Get relative position indices
        rel_pos_indices = self._get_relative_positions(seq_len).to(queries.device)  # [N, N]
        
        # Get relative position embeddings for keys
        rel_k = self.relative_position_embeddings_k(rel_pos_indices)  # [N, N, d_head]
        
        # Compute relative position attention: Q @ rel_K^T
        # queries: [B, H, N, d_head] -> [B, H, N, 1, d_head]
        # rel_k:   [N, N, d_head]    -> [1, 1, N, N, d_head]
        # We need: for each query position i, dot product with rel_k[i, j] for all j
        queries_for_rel = queries.unsqueeze(-2)  # [B, H, N, 1, d_head]
        rel_k_expanded = rel_k.unsqueeze(0).unsqueeze(0)  # [1, 1, N, N, d_head]
        
        rel_attn_logits = (queries_for_rel * rel_k_expanded).sum(-1)  # [B, H, N, N]
        
        # Combine standard and relative attention logits
        attn_logits = (attn_logits + rel_attn_logits) / math.sqrt(d_head)
        
        # Softmax
        attn_weights = torch.softmax(attn_logits, dim=-1)  # [B, H, N, N]
        
        # Standard value aggregation
        attn_output = torch.matmul(attn_weights, values)  # [B, H, N, d_head]
        
        # Add relative position contribution to values
        rel_v = self.relative_position_embeddings_v(rel_pos_indices)  # [N, N, d_head]
        # attn_weights: [B, H, N, N] -> need to weight rel_v by attention
        # rel_v: [N, N, d_head] -> [1, 1, N, N, d_head]
        rel_v_expanded = rel_v.unsqueeze(0).unsqueeze(0)
        rel_v_output = (attn_weights.unsqueeze(-1) * rel_v_expanded).sum(-2)  # [B, H, N, d_head]
        
        attn_output = attn_output + rel_v_output
        
        return attn_output, attn_weights


# --- Demonstration ---
batch_size = 2
num_heads = 4
seq_len = 20
d_model = 64
d_head = d_model // num_heads
max_rel_pos = 8  # Clip distances to [-8, 8]

rel_pe = RelativePositionEmbedding(d_model, num_heads, max_relative_position=max_rel_pos)

# Simulate Q, K, V (normally these come from linear projections)
Q = torch.randn(batch_size, num_heads, seq_len, d_head)
K = torch.randn(batch_size, num_heads, seq_len, d_head)
V = torch.randn(batch_size, num_heads, seq_len, d_head)

output, weights = rel_pe(Q, K, V)

print(f"Input Q/K/V shape: [{batch_size}, {num_heads}, {seq_len}, {d_head}]")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"\nRelative position clip range: [{-max_rel_pos}, {max_rel_pos}]")
print(f"Number of unique relative positions: {2 * max_rel_pos + 1}")
print(f"Embedding params (K): {rel_pe.relative_position_embeddings_k.weight.numel():,}")
print(f"Embedding params (V): {rel_pe.relative_position_embeddings_v.weight.numel():,}")

# Visualise the relative position matrix
rel_pos_matrix = rel_pe._get_relative_positions(seq_len)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
im = ax.imshow(rel_pos_matrix.numpy(), cmap='coolwarm', aspect='auto')
ax.set_xlabel('Key Position (j)')
ax.set_ylabel('Query Position (i)')
ax.set_title(f'Clipped Relative Position Indices\n(clip={max_rel_pos}, shifted to [0, {2*max_rel_pos}])')
plt.colorbar(im, ax=ax)

# Attention pattern from one head
ax = axes[1]
im = ax.imshow(weights[0, 0].detach().numpy(), cmap='hot', aspect='auto')
ax.set_xlabel('Key Position')
ax.set_ylabel('Query Position')
ax.set_title('Attention Weights (Head 0, Random Init)')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print("\n✓ Relative Positional Embedding (Shaw et al.) implemented")

## 5. Rotary Position Embeddings (RoPE)
*Introduced in: "RoFormer: Enhanced Transformer with Rotary Position Embedding" (Su et al., 2021)*
*Used in: LLaMA, LLaMA 2/3, Mistral, PaLM, CodeLlama, Qwen, DeepSeek, Phi, and most modern open-source LLMs*

---

### Motivation

RoPE is arguably the most elegant positional encoding method. It achieves the best of both worlds:
* Encodes **absolute** position information into each token's query/key vectors
* Makes the attention dot product depend only on **relative** position
* Does so with **zero additional parameters**

The core insight: instead of *adding* positional information to embeddings, we **rotate** the query and key vectors by an angle proportional to their position. When we compute the dot product $$q_i \cdot k_j$$, the rotation angles partially cancel, leaving a result that depends only on the relative distance $$(i - j)$$.

### Mathematical Derivation

#### Step 1: The Desired Property

We seek a function $$f(x, pos)$$ that encodes position $$pos$$ into vector $$x$$, such that the inner product between any two encoded vectors depends only on their relative position:

$$
\langle f(q, m), f(k, n) \rangle = g(q, k, m - n)
$$

for some function $$g$$ that depends on the content $$q, k$$ and the relative position $$m - n$$, but NOT on the absolute positions $$m, n$$ individually.

#### Step 2: Solution in 2D

Consider a 2D vector $$x = [x_0, x_1]^T$$. The function that satisfies our property is a **rotation**:

$$
f(x, m) = R(m\theta) \cdot x = \begin{bmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{bmatrix} \begin{bmatrix} x_0 \\ x_1 \end{bmatrix}
$$

The inner product of two rotated vectors:

$$
\langle R(m\theta)q, R(n\theta)k \rangle = q^T R(m\theta)^T R(n\theta) k = q^T R((n-m)\theta) k
$$

Since $$R(\alpha)^T R(\beta) = R(\beta - \alpha)$$, the dot product depends only on the **difference** $$(n - m)$$.

#### Step 3: Extension to d Dimensions

For a $$d$$-dimensional vector (where $$d$$ is even), we pair up dimensions and apply independent rotations with different frequencies to each pair:

$$
R_{\Theta, m} = \begin{bmatrix}
\cos(m\theta_0) & -\sin(m\theta_0) & 0 & 0 & \cdots & 0 & 0 \\
\sin(m\theta_0) & \cos(m\theta_0) & 0 & 0 & \cdots & 0 & 0 \\
0 & 0 & \cos(m\theta_1) & -\sin(m\theta_1) & \cdots & 0 & 0 \\
0 & 0 & \sin(m\theta_1) & \cos(m\theta_1) & \cdots & 0 & 0 \\
\vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots \\
0 & 0 & 0 & 0 & \cdots & \cos(m\theta_{d/2-1}) & -\sin(m\theta_{d/2-1}) \\
0 & 0 & 0 & 0 & \cdots & \sin(m\theta_{d/2-1}) & \cos(m\theta_{d/2-1})
\end{bmatrix}
$$

where the frequencies follow the same geometric progression as sinusoidal PE:

$$
\theta_i = 10000^{-2i/d}, \quad i = 0, 1, \ldots, d/2 - 1
$$

#### Step 4: Efficient Computation

We never explicitly construct the full rotation matrix. Instead, we use the identity:

$$
R_{\Theta,m} \cdot x = x \odot \cos(m\Theta) + \text{rotate\_half}(x) \odot \sin(m\Theta)
$$

where $$\odot$$ is element-wise multiplication, and $$\text{rotate\_half}$$ swaps and negates pairs:

$$
\text{rotate\_half}([x_0, x_1, x_2, x_3, \ldots, x_{d-2}, x_{d-1}]) = [-x_1, x_0, -x_3, x_2, \ldots, -x_{d-1}, x_{d-2}]
$$

#### Step 5: Application to Attention

RoPE is applied to queries and keys **after** the linear projections but **before** the dot product:

$$
q_m' = R_{\Theta, m} \cdot W_Q x_m
$$
$$
k_n' = R_{\Theta, n} \cdot W_K x_n
$$

$$
\text{Attention}(q_m', k_n') = (R_{\Theta,m} q_m)^T (R_{\Theta,n} k_n) = q_m^T R_{\Theta, n-m} k_n
$$

The attention logit between positions $$m$$ and $$n$$ depends on the relative position $$(n - m)$$.

**Note**: RoPE is NOT applied to values — only to queries and keys.

### Pros

* **Zero additional parameters** — position is encoded via rotation, not learned embeddings
* **Relative position via absolute encoding** — elegant mathematical property where absolute rotations yield relative dot products
* **Decays with distance** — the dot product between tokens naturally decays as their distance increases (due to frequency mixing), providing an inductive bias toward locality
* **Flexible sequence length** — can be computed for any position (no embedding table limit)
* **Compatible with linear attention** — the rotation can be applied to Q and K independently
* **Strong empirical performance** — dominant in modern LLMs (LLaMA, Mistral, etc.)
* **Efficient** — only element-wise operations, no matrix multiplications for the positional part

### Cons

* **Extrapolation still degrades** — despite being defined for all positions, performance drops beyond training context length (motivating techniques like NTK-aware scaling, YaRN, etc.)
* **Low-frequency dimensions are underutilised** — the geometric frequency progression means high-index dimensions rotate extremely slowly (essentially constant for short sequences)
* **Not applied to values** — positional information only flows through the attention routing, not the value aggregation
* **Complex to implement correctly** — the rotate_half operation and frequency computation require careful implementation
* **Interacts with attention scaling** — rotation doesn't preserve the variance of dot products perfectly, requiring careful initialisation

In [0]:
class RotaryPositionalEmbedding(nn.Module):
    """
    Rotary Position Embedding (RoPE) as used in LLaMA, Mistral, etc.
    
    Applies rotation to query and key vectors such that their dot product
    depends only on relative position.
    """
    
    def __init__(self, d_model: int, max_len: int = 4096, base: float = 10000.0):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        self.base = base
        
        # Compute inverse frequencies: theta_i = 1 / (base^(2i/d))
        # These determine the rotation speed for each dimension pair
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)  # [d_model/2]
        
        # Precompute cos and sin for all positions
        self._build_cache(max_len)
    
    def _build_cache(self, max_len: int):
        """Precompute rotation matrices for positions [0, max_len)."""
        positions = torch.arange(max_len, dtype=self.inv_freq.dtype)  # [max_len]
        
        # Outer product: [max_len, d_model/2]
        # freqs[pos, i] = pos * theta_i
        freqs = torch.einsum('p,d->pd', positions, self.inv_freq)  # [max_len, d/2]
        
        # Duplicate for sin and cos (we need d values total)
        # Each pair (2i, 2i+1) shares the same frequency
        emb = torch.cat([freqs, freqs], dim=-1)  # [max_len, d]
        
        self.register_buffer('cos_cached', emb.cos())  # [max_len, d]
        self.register_buffer('sin_cached', emb.sin())  # [max_len, d]
    
    @staticmethod
    def rotate_half(x: torch.Tensor) -> torch.Tensor:
        """
        Rotates half of the dimensions: [x0, x1, x2, x3, ...] -> [-x1, x0, -x3, x2, ...]
        
        This implements the multiplication by the imaginary unit in complex notation.
        """
        d = x.shape[-1]
        x1 = x[..., :d // 2]   # First half
        x2 = x[..., d // 2:]   # Second half
        return torch.cat([-x2, x1], dim=-1)
    
    def forward(self, q: torch.Tensor, k: torch.Tensor, seq_len: int = None) -> tuple:
        """
        Apply rotary embeddings to queries and keys.
        
        Args:
            q: Query tensor [batch, num_heads, seq_len, d_head]
            k: Key tensor   [batch, num_heads, seq_len, d_head]
            seq_len: Optional explicit sequence length
        Returns:
            (rotated_q, rotated_k)
        """
        if seq_len is None:
            seq_len = q.shape[2]
        
        # Get precomputed cos and sin for the required positions
        cos = self.cos_cached[:seq_len]  # [seq_len, d]
        sin = self.sin_cached[:seq_len]  # [seq_len, d]
        
        # Reshape for broadcasting: [1, 1, seq_len, d]
        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)
        
        # Apply rotation: x * cos + rotate_half(x) * sin
        q_rotated = q * cos + self.rotate_half(q) * sin
        k_rotated = k * cos + self.rotate_half(k) * sin
        
        return q_rotated, k_rotated


class MultiHeadAttentionWithRoPE(nn.Module):
    """
    Complete multi-head attention with RoPE integration.
    Shows how RoPE fits into the full attention computation.
    """
    
    def __init__(self, d_model: int, num_heads: int, max_len: int = 4096):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
        # RoPE operates on d_head dimensions
        self.rope = RotaryPositionalEmbedding(self.d_head, max_len=max_len)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input [batch, seq_len, d_model]
        Returns:
            Output [batch, seq_len, d_model]
        """
        B, N, D = x.shape
        
        # Project to Q, K, V
        q = self.W_q(x).view(B, N, self.num_heads, self.d_head).transpose(1, 2)  # [B, H, N, d_h]
        k = self.W_k(x).view(B, N, self.num_heads, self.d_head).transpose(1, 2)
        v = self.W_v(x).view(B, N, self.num_heads, self.d_head).transpose(1, 2)
        
        # Apply RoPE to Q and K (NOT V!)
        q, k = self.rope(q, k, seq_len=N)
        
        # Standard scaled dot-product attention
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v)  # [B, H, N, d_h]
        
        # Concatenate heads and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, N, D)
        return self.W_o(attn_output)


# --- Demonstration ---
d_model = 64
num_heads = 4
d_head = d_model // num_heads
seq_len = 32
batch_size = 2

rope = RotaryPositionalEmbedding(d_head, max_len=1024)

# Create dummy Q and K
q = torch.randn(batch_size, num_heads, seq_len, d_head)
k = torch.randn(batch_size, num_heads, seq_len, d_head)

q_rot, k_rot = rope(q, k)

print("=== Rotary Position Embedding (RoPE) ===")
print(f"Input Q shape: {q.shape}")
print(f"Rotated Q shape: {q_rot.shape}")
print(f"\nInverse frequencies (first 8): {rope.inv_freq[:8].numpy().round(4)}")
print(f"Wavelengths (first 8): {(2 * np.pi / rope.inv_freq[:8]).numpy().round(1)}")

# Verify the key property: dot product depends on relative position
print("\n=== Verifying Relative Position Property ===")
# Fix content vectors, vary positions
q_fixed = torch.randn(1, 1, 1, d_head).expand(1, 1, 10, d_head).clone()
k_fixed = torch.randn(1, 1, 1, d_head).expand(1, 1, 10, d_head).clone()

# Apply RoPE
q_r, k_r = rope(q_fixed, k_fixed, seq_len=10)

# Dot product between position m and position n should depend on (n-m)
print("Dot products (should depend only on distance):")
for distance in range(5):
    dots = []
    for start in range(5):
        dot = (q_r[0, 0, start] * k_r[0, 0, start + distance]).sum().item()
        dots.append(dot)
    print(f"  Distance {distance}: {[f'{d:.4f}' for d in dots]}")
    print(f"    -> All values similar? Std = {np.std(dots):.6f}")

In [0]:
# === Comprehensive RoPE Visualisation ===

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Rotation angles for different frequency bands
ax = axes[0, 0]
positions = np.arange(100)
for i, dim_idx in enumerate([0, 2, 4, 8, 12, 15]):
    theta = 1.0 / (10000 ** (2 * dim_idx / d_head))
    angles = positions * theta
    ax.plot(positions, np.sin(angles), label=f'dim pair {dim_idx}', alpha=0.8)
ax.set_xlabel('Position')
ax.set_ylabel('sin(m·θ_i)')
ax.set_title('Rotation Angles by Frequency Band')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 2. The frequency spectrum
ax = axes[0, 1]
dims = np.arange(d_head // 2)
freqs = 1.0 / (10000 ** (2 * dims / d_head))
wavelengths = 2 * np.pi / freqs
ax.semilogy(dims, wavelengths, 'b-o', markersize=3)
ax.set_xlabel('Dimension Pair Index')
ax.set_ylabel('Wavelength (positions)')
ax.set_title('RoPE Wavelength Spectrum')
ax.axhline(y=seq_len, color='r', linestyle='--', label=f'Seq len = {seq_len}')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Dot product decay with distance
ax = axes[0, 2]
# Theoretical: the dot product between RoPE-encoded identical vectors at different distances
test_len = 64
q_same = torch.randn(1, 1, 1, d_head).expand(1, 1, test_len, d_head).clone()
k_same = q_same.clone()  # Same content at all positions

test_rope = RotaryPositionalEmbedding(d_head, max_len=test_len)
q_rot_test, k_rot_test = test_rope(q_same, k_same, seq_len=test_len)

# Compute dot products from position 0 to all others
dots_from_0 = (q_rot_test[0, 0, 0:1, :] * k_rot_test[0, 0, :, :]).sum(-1).detach().numpy()
ax.plot(dots_from_0, 'b-', linewidth=2)
ax.set_xlabel('Distance from Query Position')
ax.set_ylabel('Dot Product')
ax.set_title('RoPE: Dot Product Decay with Distance\n(identical content vectors)')
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

# 4. 2D rotation visualisation
ax = axes[1, 0]
theta = 1.0 / 10000  # First frequency
circle = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--')
ax.add_patch(circle)
for pos in range(20):
    angle = pos * theta * 50  # Amplified for visibility
    x, y = np.cos(angle), np.sin(angle)
    ax.arrow(0, 0, x*0.9, y*0.9, head_width=0.05, head_length=0.02, 
             fc=plt.cm.viridis(pos/20), ec=plt.cm.viridis(pos/20))
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.set_title('2D Rotation Visualisation\n(one dimension pair, amplified)')
ax.grid(True, alpha=0.3)

# 5. Attention pattern with RoPE
ax = axes[1, 1]
mha_rope = MultiHeadAttentionWithRoPE(d_model=64, num_heads=4, max_len=128)
test_input = torch.randn(1, 32, 64)

# Get attention weights for visualisation
with torch.no_grad():
    B, N, D = test_input.shape
    q_vis = mha_rope.W_q(test_input).view(B, N, 4, 16).transpose(1, 2)
    k_vis = mha_rope.W_k(test_input).view(B, N, 4, 16).transpose(1, 2)
    q_vis, k_vis = mha_rope.rope(q_vis, k_vis, seq_len=N)
    attn_scores_vis = torch.matmul(q_vis, k_vis.transpose(-2, -1)) / math.sqrt(16)
    attn_weights_vis = torch.softmax(attn_scores_vis, dim=-1)

im = ax.imshow(attn_weights_vis[0, 0].numpy(), cmap='hot', aspect='auto')
ax.set_xlabel('Key Position')
ax.set_ylabel('Query Position')
ax.set_title('Attention Pattern with RoPE (Head 0)')
plt.colorbar(im, ax=ax)

# 6. Compare: with vs without RoPE
ax = axes[1, 2]
# Without RoPE
attn_no_rope = torch.matmul(
    mha_rope.W_q(test_input).view(1, 32, 4, 16).transpose(1, 2),
    mha_rope.W_k(test_input).view(1, 32, 4, 16).transpose(1, 2).transpose(-2, -1)
) / math.sqrt(16)
attn_no_rope_w = torch.softmax(attn_no_rope, dim=-1)

# Plot diagonal profiles (how much attention is on nearby tokens)
with_rope_diag = [attn_weights_vis[0, 0, i, max(0, i-5):i+6].mean().item() for i in range(5, 27)]
no_rope_diag = [attn_no_rope_w[0, 0, i, max(0, i-5):i+6].mean().item() for i in range(5, 27)]
ax.plot(with_rope_diag, label='With RoPE', linewidth=2)
ax.plot(no_rope_diag, label='Without RoPE', linewidth=2)
ax.set_xlabel('Query Position')
ax.set_ylabel('Mean Attention on ±5 Neighbours')
ax.set_title('Local Attention: RoPE vs No Position Encoding')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ RoPE implementation complete with visualisations")
print("\nKey takeaways:")
print("  1. Low-index dims rotate fast (capture local patterns)")
print("  2. High-index dims rotate slow (capture global patterns)")
print("  3. Dot product naturally decays with distance")
print("  4. RoPE induces locality bias in attention")

## 6. ALiBi (Attention with Linear Biases)
*Introduced in: "Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation" (Press et al., 2022)*
*Used in: BLOOM, MPT, Falcon (some variants)*

---

### Motivation

ALiBi takes the most radical approach: it uses **no positional embeddings at all**. Instead, it adds a simple, fixed **linear penalty** to attention scores based on the distance between query and key positions. The further apart two tokens are, the more their attention score is reduced.

The key insight: the model doesn't need complex positional representations. A simple distance-based bias in attention is sufficient — and it enables remarkable **length extrapolation** (training on short sequences, testing on much longer ones).

### Mathematical Formulation

Standard attention with ALiBi:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + m \cdot B\right)V
$$

where $$B$$ is the **bias matrix** defined as:

$$
B_{ij} = -|i - j|
$$

Explicitly:

$$
B = \begin{bmatrix}
0 & -1 & -2 & -3 & \cdots & -(n-1) \\
-1 & 0 & -1 & -2 & \cdots & -(n-2) \\
-2 & -1 & 0 & -1 & \cdots & -(n-3) \\
\vdots & & & & \ddots & \vdots \\
-(n-1) & -(n-2) & \cdots & & & 0
\end{bmatrix}
$$

For **causal (autoregressive)** models, the upper triangle is masked, so:

$$
B_{ij} = \begin{cases} -(i - j) & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}
$$

### Head-Specific Slopes

The scalar $$m$$ is different for each attention head, forming a geometric sequence:

$$
m_h = 2^{-8h/H}, \quad h = 1, 2, \ldots, H
$$

where $$H$$ is the total number of heads. For example, with 8 heads:

$$
m \in \left\{\frac{1}{2}, \frac{1}{4}, \frac{1}{8}, \frac{1}{16}, \frac{1}{32}, \frac{1}{64}, \frac{1}{128}, \frac{1}{256}\right\}
$$

This creates a **spectrum of locality**:
* Heads with large $$m$$ (e.g., 1/2) have strong local bias — they mostly attend to nearby tokens
* Heads with small $$m$$ (e.g., 1/256) have weak bias — they can attend globally

### Why It Enables Extrapolation

The linear bias is a simple, predictable function of distance. When the model encounters longer sequences at inference:
* The bias for distant tokens becomes more negative, but in a **smooth, predictable** way
* The softmax naturally handles larger negative values by down-weighting them
* No new positional representations need to be generated or interpolated
* The model's learned attention patterns ("attend to the last 5 tokens" for local heads) transfer seamlessly to any length

### Pros

* **Best extrapolation** — can be trained on 1024 tokens and inference on 2048+ with minimal degradation
* **Zero parameters** — the bias is fixed and computed analytically
* **Extremely simple** — just a constant matrix added to attention logits
* **Minimal computational overhead** — the bias matrix can be precomputed and cached
* **Principled multi-scale attention** — different heads naturally specialise to different distance ranges
* **Memory efficient** — no positional embedding table, no precomputed caches

### Cons

* **Weaker at modelling complex positional relationships** — the linear penalty is very simple; it cannot capture non-monotonic positional patterns
* **Strictly distance-based** — cannot differentiate between "3 tokens to the left" and "3 tokens to the right" (symmetric in bidirectional models)
* **Less popular in modern LLMs** — RoPE has become dominant; ALiBi is less studied and optimised
* **Performance gap on short contexts** — at training-length sequences, RoPE and learned embeddings typically outperform ALiBi
* **Fixed bias schedule** — the geometric slope progression is hand-designed, not learned
* **Not as effective with GQA/MQA** — fewer unique heads means fewer unique slopes, reducing the multi-scale benefit

In [0]:
class ALiBi(nn.Module):
    """
    Attention with Linear Biases (ALiBi).
    
    Adds a fixed linear distance-based bias to attention scores.
    No learnable parameters. Enables strong length extrapolation.
    """
    
    def __init__(self, num_heads: int, max_len: int = 4096):
        super().__init__()
        self.num_heads = num_heads
        
        # Compute head-specific slopes: geometric sequence
        # m_h = 2^(-8*h/H) for h = 1, ..., H
        slopes = self._get_slopes(num_heads)
        self.register_buffer('slopes', torch.tensor(slopes).float())  # [num_heads]
        
        # Precompute bias matrix for max_len
        self._build_bias_cache(max_len)
    
    @staticmethod
    def _get_slopes(num_heads: int) -> list:
        """
        Get the ALiBi slopes for each head.
        
        For num_heads that is a power of 2, uses the standard geometric sequence.
        For others, interpolates between the closest power-of-2 sequences.
        """
        def get_slopes_power_of_2(n):
            start = 2 ** (-(2 ** -(math.log2(n) - 3)))
            ratio = start
            return [start * ratio ** i for i in range(n)]
        
        if math.log2(num_heads).is_integer():
            return get_slopes_power_of_2(num_heads)
        else:
            # For non-power-of-2, use closest powers and interleave
            closest_power_of_2 = 2 ** math.floor(math.log2(num_heads))
            slopes_a = get_slopes_power_of_2(closest_power_of_2)
            slopes_b = get_slopes_power_of_2(2 * closest_power_of_2)
            # Take every other element from the larger set
            extra_slopes = slopes_b[0::2][:num_heads - closest_power_of_2]
            return slopes_a + extra_slopes
    
    def _build_bias_cache(self, max_len: int):
        """Precompute the distance-based bias matrix."""
        # Create distance matrix: bias[i, j] = -(i - j) for causal (j <= i)
        positions = torch.arange(max_len)
        # For causal attention: query at i, key at j (j <= i)
        # Distance = i - j (always non-negative for causal)
        relative_positions = positions.unsqueeze(0) - positions.unsqueeze(1)  # [max_len, max_len]
        
        # For causal: use -|i-j| where j <= i (lower triangle)
        # For bidirectional: use -|i-j| everywhere
        distance_matrix = -torch.abs(relative_positions).float()  # [max_len, max_len]
        
        # Expand for all heads: [1, num_heads, max_len, max_len]
        # Each head gets the distance matrix scaled by its slope
        alibi_bias = distance_matrix.unsqueeze(0).unsqueeze(0) * self.slopes.view(1, -1, 1, 1)
        
        self.register_buffer('bias', alibi_bias)  # [1, H, max_len, max_len]
    
    def forward(
        self, 
        q: torch.Tensor, 
        k: torch.Tensor, 
        v: torch.Tensor,
        causal: bool = True
    ) -> tuple:
        """
        Compute attention with ALiBi bias.
        
        Args:
            q: [batch, num_heads, seq_len, d_head]
            k: [batch, num_heads, seq_len, d_head]
            v: [batch, num_heads, seq_len, d_head]
            causal: Whether to apply causal mask
        Returns:
            (attention_output, attention_weights)
        """
        B, H, N, d = q.shape
        
        # Standard attention logits
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d)  # [B, H, N, N]
        
        # Add ALiBi bias (slice from precomputed cache)
        attn_scores = attn_scores + self.bias[:, :, :N, :N]
        
        # Apply causal mask if needed
        if causal:
            causal_mask = torch.triu(torch.ones(N, N, device=q.device), diagonal=1).bool()
            attn_scores = attn_scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v)
        
        return attn_output, attn_weights


# --- Demonstration ---
num_heads = 8
d_head = 64
seq_len = 32
batch_size = 1

alibi = ALiBi(num_heads=num_heads, max_len=128)

# Create dummy Q, K, V
q = torch.randn(batch_size, num_heads, seq_len, d_head)
k = torch.randn(batch_size, num_heads, seq_len, d_head)
v = torch.randn(batch_size, num_heads, seq_len, d_head)

output, weights = alibi(q, k, v, causal=True)

print("=== ALiBi (Attention with Linear Biases) ===")
print(f"Output shape: {output.shape}")
print(f"\nHead slopes: {alibi.slopes.numpy().round(6)}")
print(f"\nSlope ratios (each / previous):")
for i in range(1, num_heads):
    print(f"  Head {i}: {alibi.slopes[i].item() / alibi.slopes[i-1].item():.4f}")

# Visualise ALiBi
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. The raw bias matrix for one head
ax = axes[0, 0]
bias_head0 = alibi.bias[0, 0, :seq_len, :seq_len].numpy()
im = ax.imshow(bias_head0, cmap='RdBu_r', aspect='auto')
ax.set_xlabel('Key Position')
ax.set_ylabel('Query Position')
ax.set_title(f'ALiBi Bias Matrix (Head 0, slope={alibi.slopes[0]:.4f})')
plt.colorbar(im, ax=ax)

# 2. Bias matrices for different heads
ax = axes[0, 1]
for h in [0, 2, 4, 7]:
    # Extract the row for query position 20 (how it views keys)
    row = alibi.bias[0, h, 20, :seq_len].numpy()
    ax.plot(row, label=f'Head {h} (m={alibi.slopes[h]:.4f})', linewidth=2)
ax.set_xlabel('Key Position')
ax.set_ylabel('Bias Value')
ax.set_title('ALiBi Bias from Query Position 20')
ax.axvline(x=20, color='k', linestyle='--', alpha=0.5, label='Query pos')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3. Slopes visualisation
ax = axes[0, 2]
head_indices = np.arange(num_heads)
ax.bar(head_indices, alibi.slopes.numpy(), color=plt.cm.viridis(head_indices / num_heads))
ax.set_xlabel('Head Index')
ax.set_ylabel('Slope (m)')
ax.set_title('ALiBi Slopes per Head')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# 4. Attention patterns for different heads
for idx, head_idx in enumerate([0, 3, 7]):
    ax = axes[1, idx]
    # Apply causal mask for visualisation
    w = weights[0, head_idx].detach().numpy()
    im = ax.imshow(w, cmap='hot', aspect='auto')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')
    ax.set_title(f'Attention Pattern - Head {head_idx}\n(slope={alibi.slopes[head_idx]:.4f})')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print("\n✓ ALiBi implemented")
print("\nKey observations:")
print("  - Head 0 (steep slope): very local attention")
print("  - Head 7 (gentle slope): nearly global attention")
print("  - The geometric progression creates a spectrum of receptive fields")

## 7. Context Length Extension Techniques

A major practical challenge with positional embeddings is **extending the context window** beyond training length. Several techniques have been developed, primarily for RoPE:

### 7.1 Position Interpolation (PI)
*Chen et al., 2023 (Meta)*

Instead of extrapolating to unseen positions, **compress** the extended positions into the trained range:

$$
\theta_i' = \frac{\theta_i}{s} \quad \text{where } s = \frac{L_{\text{target}}}{L_{\text{train}}}
$$

For example, to extend from 2048 to 8192 tokens, we set $$s = 4$$ and divide all frequencies by 4. This means position 8192 gets the same encoding that position 2048 would have had, so the model sees familiar patterns.

**Trade-off**: Requires a small amount of fine-tuning, and reduces resolution (nearby positions become harder to distinguish).

### 7.2 NTK-Aware Scaling
*"Code Llama" (Rozière et al., 2023) and Reddit user "bloc97"*

Instead of scaling all frequencies uniformly, adjust the base:

$$
\theta_i' = \left(\text{base} \cdot s^{d/(d-2)}\right)^{-2i/d}
$$

This scales high frequencies less (preserving local resolution) and low frequencies more (extending global reach). The name comes from Neural Tangent Kernel theory.

### 7.3 YaRN (Yet another RoPE extensioN)
*Peng et al., 2023*

Combines NTK-aware interpolation with a temperature scaling on attention logits:

$$
\text{Attention} = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k} \cdot t}\right)V
$$

where $$t > 1$$ is a temperature that compensates for the increased entropy caused by length extension.

### 7.4 Dynamic NTK

Adapts the scaling factor dynamically based on the actual sequence length seen at inference:

$$
s = \max\left(1, \frac{L_{\text{current}}}{L_{\text{train}}}\right)
$$

No extension is applied within the training range; scaling kicks in only when needed.

In [0]:
class RoPEWithPositionInterpolation(nn.Module):
    """
    RoPE with Position Interpolation for context length extension.
    
    Scales the position indices to fit within the original training range,
    allowing inference on longer sequences.
    """
    
    def __init__(
        self, 
        d_model: int, 
        max_len: int = 4096, 
        base: float = 10000.0, 
        original_max_len: int = 2048,
        scaling_factor: float = None
    ):
        super().__init__()
        self.d_model = d_model
        self.original_max_len = original_max_len
        self.max_len = max_len
        
        # Compute scaling factor
        if scaling_factor is None:
            self.scaling_factor = max_len / original_max_len
        else:
            self.scaling_factor = scaling_factor
        
        # Standard inverse frequencies
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)
        
        self._build_cache(max_len)
    
    def _build_cache(self, max_len: int):
        """Build cache with interpolated positions."""
        # Key difference: scale positions DOWN by the scaling factor
        # Position 4096 becomes 4096/2 = 2048 (mapped back to training range)
        positions = torch.arange(max_len, dtype=self.inv_freq.dtype)
        positions_scaled = positions / self.scaling_factor  # <-- This is the interpolation!
        
        freqs = torch.einsum('p,d->pd', positions_scaled, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        
        self.register_buffer('cos_cached', emb.cos())
        self.register_buffer('sin_cached', emb.sin())
    
    @staticmethod
    def rotate_half(x):
        d = x.shape[-1]
        x1 = x[..., :d // 2]
        x2 = x[..., d // 2:]
        return torch.cat([-x2, x1], dim=-1)
    
    def forward(self, q, k, seq_len=None):
        if seq_len is None:
            seq_len = q.shape[2]
        
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        
        q_rot = q * cos + self.rotate_half(q) * sin
        k_rot = k * cos + self.rotate_half(k) * sin
        return q_rot, k_rot


class RoPEWithNTKScaling(nn.Module):
    """
    RoPE with NTK-aware scaling for context extension.
    
    Adjusts the base frequency instead of positions, preserving
    high-frequency (local) information while extending low-frequency (global) range.
    """
    
    def __init__(
        self, 
        d_model: int, 
        max_len: int = 4096, 
        base: float = 10000.0,
        original_max_len: int = 2048
    ):
        super().__init__()
        self.d_model = d_model
        
        # NTK-aware scaling: adjust the base
        scaling_factor = max_len / original_max_len
        # New base = base * scaling_factor^(d/(d-2))
        new_base = base * (scaling_factor ** (d_model / (d_model - 2)))
        
        inv_freq = 1.0 / (new_base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)
        
        # Standard cache (no position scaling needed)
        positions = torch.arange(max_len, dtype=inv_freq.dtype)
        freqs = torch.einsum('p,d->pd', positions, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos_cached', emb.cos())
        self.register_buffer('sin_cached', emb.sin())
        
        self.base = base
        self.new_base = new_base
        self.scaling_factor = scaling_factor
    
    @staticmethod
    def rotate_half(x):
        d = x.shape[-1]
        return torch.cat([-x[..., d//2:], x[..., :d//2]], dim=-1)
    
    def forward(self, q, k, seq_len=None):
        if seq_len is None:
            seq_len = q.shape[2]
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        q_rot = q * cos + self.rotate_half(q) * sin
        k_rot = k * cos + self.rotate_half(k) * sin
        return q_rot, k_rot


# --- Comparison Demonstration ---
d_head = 32
original_len = 64
extended_len = 256  # 4x extension

# Standard RoPE (no extension)
rope_standard = RotaryPositionalEmbedding(d_head, max_len=original_len)

# Position Interpolation
rope_pi = RoPEWithPositionInterpolation(
    d_head, max_len=extended_len, original_max_len=original_len
)

# NTK-aware scaling
rope_ntk = RoPEWithNTKScaling(
    d_head, max_len=extended_len, original_max_len=original_len
)

print("=== Context Length Extension Comparison ===")
print(f"Original training length: {original_len}")
print(f"Extended target length:   {extended_len} ({extended_len/original_len:.0f}x)")
print(f"\nPosition Interpolation: scales positions by 1/{extended_len/original_len:.0f}")
print(f"NTK-aware scaling: base 10000 -> {rope_ntk.new_base:.1f}")

# Compare frequency spectra
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.semilogy(rope_standard.inv_freq.numpy(), 'b-o', markersize=3, label='Standard RoPE')
ax.semilogy(rope_pi.inv_freq.numpy(), 'r-s', markersize=3, label='PI (same freqs, scaled pos)')
ax.semilogy(rope_ntk.inv_freq.numpy(), 'g-^', markersize=3, label='NTK-aware')
ax.set_xlabel('Dimension Pair Index')
ax.set_ylabel('Inverse Frequency')
ax.set_title('Frequency Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Compare cos values at different positions
ax = axes[1]
positions_to_check = np.arange(extended_len)
ax.plot(rope_pi.cos_cached[:, 0].numpy(), label='PI - dim 0', alpha=0.8)
ax.plot(rope_ntk.cos_cached[:, 0].numpy(), label='NTK - dim 0', alpha=0.8)
ax.axvline(x=original_len, color='k', linestyle='--', label=f'Training length ({original_len})')
ax.set_xlabel('Position')
ax.set_ylabel('cos(θ)')
ax.set_title('Cos Values at Dimension 0')
ax.legend()
ax.grid(True, alpha=0.3)

# Compare dot product decay
ax = axes[2]
test_vec = torch.randn(1, 1, 1, d_head)

for name, rope_variant, color in [
    ('Standard', rope_standard, 'blue'),
    ('PI', rope_pi, 'red'),
    ('NTK', rope_ntk, 'green')
]:
    max_l = original_len if name == 'Standard' else extended_len
    q_test = test_vec.expand(1, 1, max_l, d_head).clone()
    k_test = q_test.clone()
    q_r, k_r = rope_variant(q_test, k_test, seq_len=max_l)
    dots = (q_r[0, 0, 0:1] * k_r[0, 0]).sum(-1).detach().numpy()
    ax.plot(dots, label=name, color=color, alpha=0.8)

ax.axvline(x=original_len, color='k', linestyle='--', alpha=0.5, label='Training length')
ax.set_xlabel('Distance')
ax.set_ylabel('Dot Product')
ax.set_title('Dot Product Decay Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Context extension methods demonstrated")

## 8. Comprehensive Comparison

### Summary Table

| Property | Sinusoidal | Learned | Relative (Shaw) | RoPE | ALiBi |
| --- | --- | --- | --- | --- | --- |
| **Injection point** | Input embedding | Input embedding | Attention logits | Q/K rotation | Attention logits |
| **Learnable params** | 0 | $$L_{max} \times d$$ | $$(2k+1) \times d_h$$ per layer | 0 | 0 |
| **Encodes** | Absolute position | Absolute position | Relative distance | Relative (via absolute) | Relative distance |
| **Per-layer signal** | No (input only) | No (input only) | Yes | Yes | Yes |
| **Max sequence length** | Unlimited (theory) | Hard limit at $$L_{max}$$ | Clip at $$\pm k$$ | Unlimited (theory) | Unlimited |
| **Extrapolation quality** | Poor | None | Moderate | Moderate (good with PI/NTK) | Excellent |
| **Computational cost** | Negligible | Negligible | $$O(n^2 d)$$ per layer | $$O(nd)$$ per layer | $$O(n^2)$$ per layer |
| **Memory overhead** | Pre-computed buffer | Embedding table | Embedding table + bias | Pre-computed sin/cos | Pre-computed bias |
| **Used in** | Original Transformer | BERT, GPT-2, GPT-3 | Transformer-XL, T5 | LLaMA, Mistral, Phi, Qwen | BLOOM, MPT |

### Decision Guide

**Choose Learned Absolute** when:
* Your max sequence length is known and fixed (e.g., BERT's 512 tokens)
* You want simplicity and strong within-distribution performance
* You have enough training data to learn good positional patterns

**Choose RoPE** when:
* You're building a modern autoregressive LLM
* You need flexibility to extend context later (with PI/NTK/YaRN)
* You want the best balance of performance and efficiency
* You're following the LLaMA/Mistral recipe

**Choose ALiBi** when:
* Length extrapolation is your primary concern
* You want to train short and deploy long without any fine-tuning
* You can tolerate slightly lower performance at training-length sequences
* Simplicity of implementation is paramount

**Choose Relative (T5-style)** when:
* You're building an encoder-decoder model
* You need explicit control over how the model uses position
* You want per-head, per-layer position specialisation

In [0]:
# === Final Comparison: All Methods Side-by-Side ===

def compute_attention_with_method(method_name, seq_len=32, d_head=32, num_heads=4):
    """Compute attention weights using different positional encoding methods."""
    torch.manual_seed(42)
    
    # Same Q, K, V content for fair comparison
    q = torch.randn(1, num_heads, seq_len, d_head)
    k = torch.randn(1, num_heads, seq_len, d_head)
    v = torch.randn(1, num_heads, seq_len, d_head)
    
    if method_name == 'none':
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_head)
        
    elif method_name == 'sinusoidal':
        # Add sinusoidal PE to the "input" (before Q, K projection - simulated)
        pe = SinusoidalPositionalEncoding(d_head, max_len=seq_len, dropout=0.0)
        pe_vals = pe.pe[:, :seq_len, :]  # [1, seq_len, d_head]
        pe_vals = pe_vals.unsqueeze(1).expand(-1, num_heads, -1, -1)
        q_pe = q + pe_vals
        k_pe = k + pe_vals
        scores = torch.matmul(q_pe, k_pe.transpose(-2, -1)) / math.sqrt(d_head)
        
    elif method_name == 'rope':
        rope = RotaryPositionalEmbedding(d_head, max_len=seq_len)
        q_rot, k_rot = rope(q, k, seq_len=seq_len)
        scores = torch.matmul(q_rot, k_rot.transpose(-2, -1)) / math.sqrt(d_head)
        
    elif method_name == 'alibi':
        alibi_module = ALiBi(num_heads=num_heads, max_len=seq_len)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_head)
        scores = scores + alibi_module.bias[:, :, :seq_len, :seq_len]
    
    # Apply causal mask
    causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    
    return weights


# Generate attention patterns
methods = ['none', 'sinusoidal', 'rope', 'alibi']
titles = ['No Position Info', 'Sinusoidal PE', 'RoPE', 'ALiBi']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for idx, (method, title) in enumerate(zip(methods, titles)):
    weights = compute_attention_with_method(method)
    
    # Head 0 pattern
    ax = axes[0, idx]
    im = ax.imshow(weights[0, 0].detach().numpy(), cmap='hot', aspect='auto')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')
    ax.set_title(f'{title}\n(Head 0)')
    plt.colorbar(im, ax=ax)
    
    # Average attention distance per query position
    ax = axes[1, idx]
    positions = torch.arange(32).float()
    for h in range(4):
        w = weights[0, h].detach()
        # Compute average attention distance for each query position
        avg_dist = []
        for qi in range(32):
            key_positions = positions[:qi+1]  # causal: only attend to past
            attended_pos = (w[qi, :qi+1] * key_positions).sum()
            avg_dist.append((qi - attended_pos).item())  # average distance back
        ax.plot(avg_dist, alpha=0.6, label=f'Head {h}')
    ax.set_xlabel('Query Position')
    ax.set_ylabel('Avg Attention Distance (tokens back)')
    ax.set_title(f'{title}\nAttention Span')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Positional Encoding Methods: Attention Pattern Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Parameter comparison
print("\n" + "="*60)
print("PARAMETER COUNT COMPARISON (d_model=512, max_len=2048, 8 heads)")
print("="*60)
d = 512
L = 2048
H = 8
print(f"  Sinusoidal:    0 parameters (pre-computed)")
print(f"  Learned:       {L*d:>10,} parameters ({L}×{d})")
print(f"  Relative (k=128): {2 * (2*128+1) * (d//H) * H:>10,} parameters per layer")
print(f"  RoPE:          0 parameters (pre-computed sin/cos)")
print(f"  ALiBi:         0 parameters (fixed slopes)")

print("\n" + "="*60)
print("EXTRAPOLATION COMPARISON (trained on 2048, tested at 8192)")
print("="*60)
print("  Sinusoidal:    Defined but performance degrades significantly")
print("  Learned:       FAILS - no embedding exists for pos > 2048")
print("  Relative:      Moderate - clipping helps but degradation occurs")
print("  RoPE:          Degrades without extension; excellent with PI/NTK")
print("  ALiBi:         Best out-of-box extrapolation")

print("\n✓ All positional embedding methods compared")

## 9. Conclusion

### The Evolution of Positional Encoding

The history of positional embeddings in Transformers reflects a broader trend in deep learning: moving from **rigid, human-designed** components toward **flexible, principled** mechanisms that provide useful inductive biases without overly constraining the model.

$$
\text{Sinusoidal} \xrightarrow{\text{task adaptation}} \text{Learned} \xrightarrow{\text{relative awareness}} \text{Relative/T5} \xrightarrow{\text{elegance + efficiency}} \text{RoPE} \xrightarrow{\text{extrapolation}} \text{ALiBi}
$$

### Current State of the Art (2024-2025)

**RoPE dominates modern LLMs** for several practical reasons:
* It provides a good balance between performance and length flexibility
* Context extension techniques (PI, NTK, YaRN) have solved RoPE's extrapolation weakness
* It's well-understood, well-optimised, and widely available in frameworks
* Models like LLaMA 3 (128K context), Mistral, Qwen, and DeepSeek all use RoPE variants

### Open Questions

1. **Is position encoding necessary at all?** Some evidence suggests that causal masking alone provides sufficient positional signal for autoregressive models
2. **Can we combine methods?** E.g., RoPE for local structure + ALiBi for long-range decay
3. **What is the optimal frequency spectrum?** The $$10000^{2i/d}$$ geometric progression is somewhat arbitrary
4. **How do positional embeddings interact with KV caching, speculative decoding, and other inference optimisations?**

---

### References

1. Vaswani et al. (2017). "Attention Is All You Need." *NeurIPS*.
2. Devlin et al. (2019). "BERT: Pre-training of Deep Bidirectional Transformers." *NAACL*.
3. Shaw et al. (2018). "Self-Attention with Relative Position Representations." *NAACL*.
4. Dai et al. (2019). "Transformer-XL: Attentive Language Models Beyond a Fixed-Length Context." *ACL*.
5. Su et al. (2021). "RoFormer: Enhanced Transformer with Rotary Position Embedding." *arXiv*.
6. Press et al. (2022). "Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation." *ICLR*.
7. Raffel et al. (2020). "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer." *JMLR*.
8. Chen et al. (2023). "Extending Context Window of Large Language Models via Positional Interpolation." *arXiv*.
9. Peng et al. (2023). "YaRN: Efficient Context Window Extension of Large Language Models." *arXiv*.